# Stage 3: DPO Preference Alignment

**Project:** IT Helpdesk AI Assistant (domain-specific fine-tuning with Unsloth)

**Goal:** Improve the response quality of the instruction fine-tuned (SFT) model using Direct Preference Optimization (DPO) on `data/preference_dataset.jsonl`.

> **Run this notebook on a GPU runtime** (Google Colab free T4, or Kaggle GPU).

Steps covered:
1. Load the SFT model (from Stage 2)
2. Load preference dataset
3. Format prompt / chosen / rejected
4. Configure DPO training
5. Run DPO alignment
6. Save the DPO-aligned model
7. Test the model after DPO

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 1. Load the SFT model

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LENGTH = 1024
SFT_ADAPTER = "models/sft_adapter"
BASE_MODEL = "unsloth/tinyllama-bnb-4bit"

model_name = SFT_ADAPTER if os.path.isdir(SFT_ADAPTER) else BASE_MODEL

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 2. Load preference dataset and format prompt/chosen/rejected

`data/preference_dataset.jsonl` contains `{"prompt": ..., "chosen": ..., "rejected": ...}` records, as required by `trl.DPOTrainer`.

In [ ]:
import json
from datasets import Dataset

PROMPT_TEMPLATE = """Below is an IT Helpdesk support request. Write a helpful, professional response.

### Request:
{prompt}

### Response:
"""

records = []
with open("../data/preference_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} preference examples")

def format_dpo(example):
    return {
        "prompt": PROMPT_TEMPLATE.format(prompt=example["prompt"]),
        "chosen": example["chosen"] + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = Dataset.from_list(records).map(format_dpo)
print(dpo_dataset[0])

## 3. Configure and run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="outputs_dpo",
    beta=0.1,
    max_prompt_length=512,
    max_length=1024,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth reuses the base model as the implicit reference
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

dpo_trainer.train()

## 4. Save the DPO-aligned model

In [ ]:
SAVE_DIR = "models/dpo_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved DPO-aligned adapter to {SAVE_DIR}")

# Optional: merge LoRA weights into the base model for a standalone deployable model
# model.save_pretrained_merged("models/dpo_merged", tokenizer, save_method="merged_16bit")

## 5. Test the model after DPO

Re-run the same 10 evaluation questions to populate `reports/final_evaluation.md`.

In [ ]:
import json
import sys

sys.path.append("../src")
from eval_questions import EVAL_QUESTIONS

FastLanguageModel.for_inference(model)

dpo_answers = []
for q in EVAL_QUESTIONS:
    prompt = PROMPT_TEMPLATE.format(prompt=q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("### Response:")[-1].strip()
    dpo_answers.append({"question": q, "answer": answer})
    print("Q:", q)
    print("A:", answer)
    print("-" * 80)

with open("../reports/dpo_model_answers.json", "w", encoding="utf-8") as f:
    json.dump(dpo_answers, f, indent=2)
print("Saved DPO answers to reports/dpo_model_answers.json")